<a href="https://colab.research.google.com/github/sidharthh-a/memory-profiling/blob/main/memory_profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [ ]:
file_path = "uber-raw-data-apr14.csv"
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "fivethirtyeight/uber-pickups-in-new-york-city",
    file_path,
)

print("Loaded Data:")
print(df.head())

/tmp/ipython-input-1462181073.py:2: DeprecationWarning: load_dataset is deprecated and will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 3.45M/3.45M [00:00<00:00, 108MB/s]

Extracting zip of uber-raw-data-apr14.csv...


Loaded Data:
          Date/Time      Lat      Lon    Base
0  4/1/2014 0:11:00  40.7690 -73.9549  B02512
1  4/1/2014 0:17:00  40.7267 -74.0345  B02512
2  4/1/2014 0:21:00  40.7316 -73.9873  B02512
3  4/1/2014 0:28:00  40.7588 -73.9776  B02512
4  4/1/2014 0:33:00  40.7594 -73.9722  B02512


saving in different formats

In [ ]:
df.to_json('uber.json', orient='records', lines=True)
df.to_parquet('uber.parquet')
df.to_feather('uber.feather')

identifying continous variables

In [ ]:
cont_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
print("Continuous Variables:", cont_cols)

Continuous Variables: ['Lat', 'Lon']


defining load functions for profiling

In [ ]:
import pandas as pd

def load_json():
    return pd.read_json('uber.json', orient='records', lines=True)

def load_parquet():
    return pd.read_parquet('uber.parquet')

def load_feather():
    return pd.read_feather('uber.feather')

def load_parquet_cont():
    return pd.read_parquet('uber.parquet', columns=cont_cols)

def load_feather_cont():
    df_all = pd.read_feather('uber.feather')
    return df_all[cont_cols]


profile using timeit and memory_profiler

In [ ]:
!pip install -q memory-profiler

In [ ]:
import timeit
from memory_profiler import memory_usage

def profile_loader(func, label):
    time = timeit.timeit(func, number=5) / 5
    mem = max(memory_usage(func))
    print(f"{label:<15} | Time: {time:.6f} sec | Peak Memory: {mem:.2f} MiB")

# Run profiling
profile_loader(load_json, 'JSON')
profile_loader(load_parquet, 'Parquet')
profile_loader(load_feather, 'Feather')
profile_loader(load_parquet_cont, 'Parquet-cont')
profile_loader(load_feather_cont, 'Feather-cont')


JSON            | Time: 2.348508 sec | Peak Memory: 890.99 MiB
Parquet         | Time: 0.114146 sec | Peak Memory: 410.50 MiB
Feather         | Time: 0.076837 sec | Peak Memory: 420.12 MiB
Parquet-cont    | Time: 0.011911 sec | Peak Memory: 420.40 MiB
Feather-cont    | Time: 0.084094 sec | Peak Memory: 429.19 MiB


as per the result, parquet is the most efficient stream for continous variables.